# 타이타닉 생존 예측 - 3단계: ML 모델링 및 평가

## 프로젝트 진행 상황
- ✅ **1단계 완료**: EDA와 베이스라인 모델 (정확도: ~78%)
- ✅ **2단계 완료**: 데이터 전처리 및 특성 엔지니어링
- 🔄 **3단계 진행중**: ML 모델링 및 평가 ← **지금 여기**
- ⏳ **4단계 예정**: 최종 모델 선택 및 제출

## 🎯 이번 노트북의 목표
1. **전처리된 데이터 로딩** - 이전 단계에서 저장한 데이터 불러오기
2. **다양한 ML 알고리즘 비교** - 로지스틱 회귀, 랜덤 포레스트, SVM, XGBoost 등
3. **베이스라인 모델과 비교** - 간단한 규칙(78%)보다 얼마나 개선되었는지
4. **교차검증** - 모델의 안정성 확인
5. **하이퍼파라미터 튜닝** - 최적의 성능 찾기
6. **특성 중요도 분석** - 어떤 특성이 중요한지 파악

> 💡 **실무 팁**: 여러 모델을 비교하면서 점진적으로 개선하는 것이 핵심입니다!

## 1단계: 환경 설정 및 데이터 로딩

전처리된 데이터를 불러오고 ML 모델링 준비를 합니다.

In [ ]:
# 필수 라이브러리 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 머신러닝 모델들
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# 평가 지표
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score, GridSearchCV

# 시각화 설정
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ 라이브러리 로딩 완료!")
print("🤖 머신러닝 모델링 준비 완료!")

In [ ]:
# 전처리된 데이터 로딩
PROCESSED_PATH = '../data/processed/'

print("📂 전처리된 데이터 로딩 중...")

X_train = pd.read_csv(PROCESSED_PATH + 'X_train.csv')
X_val = pd.read_csv(PROCESSED_PATH + 'X_val.csv')
y_train = pd.read_csv(PROCESSED_PATH + 'y_train.csv').values.ravel()  # 1D array로 변환
y_val = pd.read_csv(PROCESSED_PATH + 'y_val.csv').values.ravel()
X_test = pd.read_csv(PROCESSED_PATH + 'X_test.csv')

print("✅ 데이터 로딩 완료!")
print(f"- 훈련 데이터: {X_train.shape} / 타겟: {y_train.shape}")
print(f"- 검증 데이터: {X_val.shape} / 타겟: {y_val.shape}")
print(f"- 테스트 데이터: {X_test.shape}")

print(f"\n📊 사용 특성: {X_train.shape[1]}개")
print("특성 목록:")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:2d}. {col}")

## 2단계: 여러 ML 모델 비교

실무에서는 먼저 여러 알고리즘을 빠르게 비교해서 어떤 것이 가장 적합한지 파악합니다.

In [ ]:
# 다양한 모델 정의 (실무에서 자주 사용하는 모델들)
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100),
    'SVM': SVC(random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

# 모델 학습 및 평가
results = []

print("🤖 모델 학습 및 평가 시작...\n")
print("="*80)

for name, model in models.items():
    print(f"\n📊 {name} 학습 중...")
    
    # 모델 학습
    model.fit(X_train, y_train)
    
    # 예측
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    # 평가
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    
    # 교차검증 (5-fold)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    results.append({
        'Model': name,
        'Train Acc': train_acc,
        'Val Acc': val_acc,
        'CV Mean': cv_mean,
        'CV Std': cv_std
    })
    
    print(f"  - 훈련 정확도: {train_acc:.4f}")
    print(f"  - 검증 정확도: {val_acc:.4f}")
    print(f"  - 교차검증 평균: {cv_mean:.4f} (+/- {cv_std:.4f})")
    
    # 과적합 체크
    if train_acc - val_acc > 0.05:
        print(f"  ⚠️  과적합 가능성 (훈련 정확도 >> 검증 정확도)")

print("\n" + "="*80)
print("✅ 모든 모델 학습 완료!")

In [ ]:
# 결과 정리 및 시각화
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Val Acc', ascending=False)

print("\n📊 모델 성능 비교표:")
print("="*80)
print(results_df.to_string(index=False))

# 베이스라인과 비교
baseline_acc = 0.787  # 1단계에서 만든 규칙 기반 모델의 정확도
print(f"\n🎯 베이스라인 모델 (규칙 기반): {baseline_acc:.3f}")
print(f"✨ 최고 성능 모델: {results_df.iloc[0]['Model']} - {results_df.iloc[0]['Val Acc']:.3f}")
improvement = results_df.iloc[0]['Val Acc'] - baseline_acc
print(f"📈 개선도: {improvement:+.3f} ({improvement/baseline_acc*100:+.1f}%)")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. 검증 정확도 비교
ax1 = axes[0]
colors = ['green' if acc > baseline_acc else 'orange' for acc in results_df['Val Acc']]
bars = ax1.barh(results_df['Model'], results_df['Val Acc'], color=colors, alpha=0.7)
ax1.axvline(x=baseline_acc, color='red', linestyle='--', linewidth=2, label=f'Baseline ({baseline_acc:.3f})')
ax1.set_xlabel('Validation Accuracy')
ax1.set_title('모델별 검증 정확도 비교')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# 값 표시
for i, (idx, row) in enumerate(results_df.iterrows()):
    ax1.text(row['Val Acc'], i, f" {row['Val Acc']:.3f}", va='center')

# 2. 교차검증 점수 (에러바 포함)
ax2 = axes[1]
ax2.barh(results_df['Model'], results_df['CV Mean'], 
         xerr=results_df['CV Std'], color='skyblue', alpha=0.7, capsize=5)
ax2.axvline(x=baseline_acc, color='red', linestyle='--', linewidth=2, label=f'Baseline ({baseline_acc:.3f})')
ax2.set_xlabel('Cross-Validation Accuracy')
ax2.set_title('교차검증 정확도 (5-Fold CV)')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 해석:")
print(f"- 녹색 막대: 베이스라인보다 성능 향상")
print(f"- 주황색 막대: 베이스라인보다 성능 저하")
print(f"- 에러바: 교차검증 표준편차 (작을수록 안정적)")

## 3단계: 최고 성능 모델 상세 분석

검증 정확도가 가장 높은 모델을 선택해서 자세히 분석해보겠습니다.

In [ ]:
# 최고 성능 모델 선택
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

print(f"🏆 최고 성능 모델: {best_model_name}")
print(f"검증 정확도: {results_df.iloc[0]['Val Acc']:.4f}")
print("\n" + "="*80)

# 검증 데이터에 대한 예측
y_val_pred = best_model.predict(X_val)
y_val_pred_proba = best_model.predict_proba(X_val)[:, 1] if hasattr(best_model, 'predict_proba') else None

# 상세 평가 지표
print("\n📊 상세 평가 지표:")
print(classification_report(y_val, y_val_pred, target_names=['사망', '생존']))

# 혼동 행렬
cm = confusion_matrix(y_val, y_val_pred)
print("🔍 혼동 행렬:")
print("              예측 사망   예측 생존")
print(f"실제 사망       {cm[0,0]:4d}        {cm[0,1]:4d}")
print(f"실제 생존       {cm[1,0]:4d}        {cm[1,1]:4d}")

# 혼동 행렬 시각화
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['사망', '생존'],
            yticklabels=['사망', '생존'])
plt.title(f'{best_model_name} - 혼동 행렬')
plt.ylabel('실제값')
plt.xlabel('예측값')
plt.show()

print(f"\n💡 혼동 행렬 해석:")
print(f"- 올바른 예측: {cm[0,0] + cm[1,1]}개 / {len(y_val)}개")
print(f"- 거짓 양성(False Positive): {cm[0,1]}개 - 사망했는데 생존으로 예측")
print(f"- 거짓 음성(False Negative): {cm[1,0]}개 - 생존했는데 사망으로 예측")

## 4단계: 특성 중요도 분석

어떤 특성이 생존 예측에 가장 중요한지 파악합니다. (트리 기반 모델인 경우)

In [ ]:
# 특성 중요도 분석 (트리 기반 모델만 가능)
if hasattr(best_model, 'feature_importances_'):
    print("📊 특성 중요도 분석")
    print("="*80)
    
    # 특성 중요도 추출
    feature_importance = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\n특성별 중요도 순위:")
    for idx, row in feature_importance.iterrows():
        print(f"{row['Feature']:30s}: {row['Importance']:.4f} {'■' * int(row['Importance'] * 100)}")
    
    # 시각화
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue', alpha=0.7)
    plt.xlabel('Importance Score')
    plt.title(f'{best_model_name} - 특성 중요도')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 해석:")
    top3_features = feature_importance.head(3)['Feature'].tolist()
    print(f"- 가장 중요한 특성 Top 3: {', '.join(top3_features)}")
    print(f"- 이 특성들이 생존 예측에 가장 큰 영향을 미칩니다")
    
else:
    print(f"⚠️  {best_model_name}는 특성 중요도를 제공하지 않습니다.")
    print("트리 기반 모델(Random Forest, Gradient Boosting 등)에서만 확인 가능합니다.")
    
    # 대신 Random Forest로 특성 중요도 확인
    print("\n대신 Random Forest로 특성 중요도를 확인하겠습니다...")
    rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
    rf_model.fit(X_train, y_train)
    
    feature_importance = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue', alpha=0.7)
    plt.xlabel('Importance Score')
    plt.title('Random Forest - 특성 중요도')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5단계: 하이퍼파라미터 튜닝

최고 성능 모델의 하이퍼파라미터를 조정해서 성능을 더 개선해보겠습니다.

In [ ]:
# 모델별 하이퍼파라미터 그리드 정의 (실무에서 자주 사용하는 범위)
param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'min_samples_split': [2, 5]
    },
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10, 100],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}

# 최고 성능 모델에 대해서만 튜닝 (시간 절약)
if best_model_name in param_grids:
    print(f"⚙️  {best_model_name} 하이퍼파라미터 튜닝 시작...")
    print("="*80)
    print("⏰ 시간이 걸릴 수 있습니다 (Grid Search 수행 중)...\n")
    
    # Grid Search 수행
    grid_search = GridSearchCV(
        estimator=models[best_model_name],
        param_grid=param_grids[best_model_name],
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    # 최적 파라미터와 성능
    print("\n✅ Grid Search 완료!")
    print(f"\n🏆 최적 하이퍼파라미터:")
    for param, value in grid_search.best_params_.items():
        print(f"  - {param}: {value}")
    
    print(f"\n📊 성능 비교:")
    print(f"  - 기본 모델 검증 정확도: {results_df.iloc[0]['Val Acc']:.4f}")
    
    # 튜닝된 모델로 검증
    tuned_model = grid_search.best_estimator_
    y_val_pred_tuned = tuned_model.predict(X_val)
    tuned_val_acc = accuracy_score(y_val, y_val_pred_tuned)
    
    print(f"  - 튜닝 후 검증 정확도: {tuned_val_acc:.4f}")
    improvement = tuned_val_acc - results_df.iloc[0]['Val Acc']
    print(f"  - 개선도: {improvement:+.4f}")
    
    if improvement > 0:
        print(f"\n✨ 튜닝으로 {improvement:.4f} 성능 향상!")
        best_model = tuned_model  # 최고 모델 업데이트
    else:
        print(f"\n💡 튜닝으로 큰 개선이 없습니다. 기본 파라미터도 충분히 좋습니다!")
        
else:
    print(f"⚠️  {best_model_name}에 대한 튜닝 그리드가 정의되지 않았습니다.")
    print("Random Forest, Gradient Boosting, Logistic Regression 중 하나를 선택하세요.")

## 6단계: 최종 모델로 테스트 데이터 예측

최고 성능 모델(튜닝 후)을 사용해서 실제 테스트 데이터를 예측합니다.

In [ ]:
# 테스트 데이터 예측
print("🎯 최종 모델로 테스트 데이터 예측")
print("="*80)

# 전체 훈련 데이터로 재학습 (검증 데이터도 포함)
print("\n📚 전체 훈련 데이터로 모델 재학습 중...")
X_train_full = pd.concat([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

best_model.fit(X_train_full, y_train_full)
print("✅ 재학습 완료!")

# 테스트 데이터 예측
print("\n🔮 테스트 데이터 예측 중...")
test_predictions = best_model.predict(X_test)

print(f"✅ 예측 완료!")
print(f"- 예측 건수: {len(test_predictions)}건")
print(f"- 생존 예측: {test_predictions.sum()}명 ({test_predictions.mean()*100:.1f}%)")
print(f"- 사망 예측: {(1-test_predictions).sum()}명 ({(1-test_predictions.mean())*100:.1f}%)")

# 제출 파일 생성
test_ids = pd.read_csv(PROCESSED_PATH + 'test_ids.csv')
submission = pd.DataFrame({
    'PassengerId': test_ids['PassengerId'],
    'Survived': test_predictions
})

# 저장
import os
submission_path = '../submissions/'
os.makedirs(submission_path, exist_ok=True)
submission_file = submission_path + f'submission_{best_model_name.replace(" ", "_")}.csv'
submission.to_csv(submission_file, index=False)

print(f"\n💾 제출 파일 저장 완료!")
print(f"파일 위치: {submission_file}")
print(f"\n제출 파일 미리보기:")
print(submission.head(10))

---

## 🎉 모델링 및 평가 완료!

### ✅ 완료된 작업들:

#### 1. **다양한 ML 모델 비교** ✓
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- SVM
- KNN

#### 2. **성능 평가** ✓
- 훈련/검증 정확도 비교
- 교차검증 (5-Fold CV)
- 베이스라인 모델과 비교

#### 3. **최고 모델 선택 및 분석** ✓
- 혼동 행렬 분석
- 정밀도, 재현율, F1 스코어
- 특성 중요도 분석

#### 4. **하이퍼파라미터 튜닝** ✓
- Grid Search로 최적 파라미터 탐색
- 성능 개선 확인

#### 5. **테스트 데이터 예측** ✓
- 최종 모델로 예측
- 제출 파일 생성

### 📊 최종 결과 요약

```
🎯 베이스라인 모델 (규칙 기반): ~78.7%
🤖 최고 ML 모델: [실행 후 확인]
📈 개선도: [실행 후 확인]
```

### 🚀 다음 단계 (선택사항)

1. **앙상블 기법**: 여러 모델을 결합해서 성능 향상
2. **더 많은 특성 엔지니어링**: 새로운 특성 아이디어 시도
3. **스태킹**: 모델들을 계층적으로 결합
4. **Kaggle 제출**: 실제 리더보드에서 점수 확인

### 💡 실무 인사이트

**이번 노트북에서 배운 실무 스킬:**
- 여러 알고리즘을 체계적으로 비교하는 방법
- 교차검증으로 모델 안정성 평가
- 특성 중요도로 모델 해석
- 하이퍼파라미터 튜닝으로 성능 최적화
- 과적합 감지 및 방지

**실무 팁**: 
- 항상 베이스라인부터 시작해서 점진적으로 개선
- 복잡한 모델이 항상 좋은 것은 아님
- 교차검증으로 일반화 성능 확인 필수
- 특성 중요도로 모델을 이해하고 설명

---

**축하합니다! 🎊**  
체계적인 ML 프로젝트 워크플로우를 모두 경험해보셨습니다!
1단계(EDA) → 2단계(전처리) → 3단계(모델링) 순서로 진행하는 것이 실무의 표준입니다.